# 22장. 통합과 리포트 작성

| 항목 | 내용 |
|---|---|
| 예상 시간 | 4시간 |
| 선수 장 | 1장, 2장, 21장까지 전체 |
| 이 장의 결과물 | `src/` 모듈 구조와 `main.py`, 테스트, README(필수 3단 구성), 제출 전 체크리스트 |
| 과제 연결 | 최종 결과물 1~4 전체, 실행 방법 명시(필수), `requirements.txt` 제출 |

**학습 목표**
- 노트북에서 만든 코드를 `src/` 모듈과 `main.py` 진입점으로 나눌 수 있다.
- 결과물을 정해진 규칙으로 저장하고, 테스트로 수정 후에도 결과가 유지되는지 확인할 수 있다.
- 과제가 요구하는 README 3단 구성(구현 요약 → 검증·실험 결과 → 한계와 개선 방향)을 작성할 수 있다.
- 재현성(시드, 설정, 버전)을 갖추고, 제출 전 체크리스트로 빠진 항목을 찾을 수 있다.

이 장은 실제 데이터 대신 **임시 폴더에 만든 미니 프로젝트**로 구조를 연습하고, 마지막에 실제 프로젝트 폴더를 **읽기만 해서** 제출 조건을 점검합니다.

In [1]:
import ast
import os
import re
import subprocess
import sys
import tempfile
from importlib.metadata import version
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "studys" else Path.cwd()
ENV = {**os.environ, "PYTHON_COLORS": "0", "NO_COLOR": "1"}   # 하위 프로세스 출력의 색상 코드 끄기


def run(args: list[str], cwd: Path) -> None:
    """cwd에서 파이썬을 실행하고 출력을 보여 준다 (임시 폴더 경로는 출력에서 지운다)."""
    result = subprocess.run([sys.executable, *args], cwd=cwd, capture_output=True, text=True, env=ENV)
    out = (result.stdout + result.stderr).replace(str(cwd) + os.sep, "").strip()
    print(f"$ python {' '.join(args)}\n{out}\n")

## 22.1 모듈 나누기와 `main.py` 진입점

노트북은 실험하기 좋지만, 제출물로는 부족합니다. 평가자가 **명령 한 줄로 실행**할 수 있어야 하고, 같은 코드를 검색·분류·테스트가 함께 써야 하기 때문입니다(1장 1.2절).

### 옮기는 원칙
1. **한 모듈은 한 단계**를 맡습니다. 장 구성과 거의 같습니다.
2. 모듈에는 **함수와 클래스만** 둡니다. 실행 코드는 `if __name__ == "__main__":` 아래나 `main.py`에 둡니다.
3. 숫자 설정(시드, `min_df`, 불용어 목록)은 모듈 맨 위 **상수**로 모아 근거를 주석으로 남깁니다.
4. 노트북에서 확인한 계산은 **테스트**로 고정합니다.

| 장 | 모듈 | 내용 |
|---|---|---|
| 7장 | `src/data.py` | 데이터 로드, 개인정보 가리기, 중복 제거 |
| 8~10장 | `src/preprocess.py` | 정제, 토큰화, 불용어 |
| 10~11장 | `src/corpus.py` | 로드 + 전처리, 캐시, `min_df` |
| 11~13장 | `src/tfidf.py` | `NumpyTfidf` |
| 14장 | `src/verify.py` | sklearn 비교 검증 로그 |
| 15~16장 | `src/search.py` | 코사인 유사도, 검색, 스니펫 |
| 17~20장 | `src/classify.py`, `src/error_analysis.py` | 분류, 평가, 혼동 행렬, 오분류 분석 |
| 21장 | `src/bow_limits.py` | BoW 한계 사례 |
| 전체 | `main.py` | CLI 진입점 |

### 미니 프로젝트로 연습하기
임시 폴더에 같은 구조의 작은 프로젝트를 만듭니다. 내용은 지금까지 만든 코드를 줄인 것입니다(`src/tfidf.py`는 13장의 `NumpyTfidf` 그대로).

In [2]:
# 파일 경로 → 내용. 각 파일은 지금까지의 장에서 만든 코드를 줄인 것이다.
MINI_SRC = {
    "src/__init__.py": r'''''',
    "src/data.py": r'''"""미니 코퍼스 (실제 프로젝트에서는 20 Newsgroups, KLUE-YNAT을 불러온다)."""

DOCS = [
    "The rocket launched into space!",
    "NASA launched a new space telescope.",
    "The pitcher threw a fast ball.",
    "Our team won the baseball game.",
    "Astronauts watched the baseball game in space.",
    "The hockey team lost the game in overtime.",
    "A new telescope image of the galaxy was released.",
    "Fans cheered as the pitcher struck out the batter.",
]
''',
    "src/preprocess.py": r'''"""정제 → 토큰화 → 불용어 제거 (8~10장을 단순화한 버전)."""
import re

STOPWORDS = frozenset({"the", "a", "an", "into", "our", "in", "new", "of", "as", "was", "out"})
TOKEN = re.compile(r"[a-z]+")


def preprocess(text: str) -> list[str]:
    return [t for t in TOKEN.findall(text.lower()) if t not in STOPWORDS and len(t) >= 2]
''',
    "src/tfidf.py": r'''import numpy as np

# ── 13장에서 만든 NumPy TF-IDF (src/tfidf.py로 옮길 내용) ──
class NumpyTfidf:
    """TF(raw 또는 log) × smooth IDF, 문서마다 L2 정규화. sklearn TfidfVectorizer와 같은 정의."""

    def __init__(self, tf: str = "raw", smooth_idf: bool = True, norm: str | None = "l2",
                 min_df: int = 1, max_df: float = 1.0):
        if tf not in ("raw", "log"):
            raise ValueError(f"tf는 'raw' 또는 'log': {tf}")
        if norm not in ("l2", None):
            raise ValueError(f"norm은 'l2' 또는 None: {norm}")
        self.tf, self.smooth_idf, self.norm = tf, smooth_idf, norm
        self.min_df, self.max_df = min_df, max_df

    def fit(self, docs: list[list[str]]) -> "NumpyTfidf":
        n = len(docs)
        df: dict[str, int] = {}
        for doc in docs:
            for t in set(doc):
                df[t] = df.get(t, 0) + 1
        max_count = self.max_df * n if isinstance(self.max_df, float) else self.max_df
        terms = sorted(t for t, c in df.items() if self.min_df <= c <= max_count)
        self.vocabulary_ = {t: i for i, t in enumerate(terms)}
        self.terms_ = np.array(terms, dtype=object)
        self.n_docs_ = n
        self.df_ = np.array([df[t] for t in terms], dtype=np.float64)
        if self.smooth_idf:
            self.idf_ = np.log((1.0 + n) / (1.0 + self.df_)) + 1.0
        else:
            self.idf_ = np.log(n / self.df_) + 1.0
        return self

    def count_matrix(self, docs: list[list[str]]) -> np.ndarray:
        rows, cols = [], []
        for i, doc in enumerate(docs):
            for t in doc:
                j = self.vocabulary_.get(t)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        counts = np.zeros((len(docs), len(self.vocabulary_)), dtype=np.float64)
        np.add.at(counts, (np.asarray(rows, dtype=np.int64), np.asarray(cols, dtype=np.int64)), 1.0)
        return counts

    def transform(self, docs: list[list[str]]) -> np.ndarray:
        x = self.count_matrix(docs)
        if self.tf == "log":
            mask = x > 0
            x[mask] = 1.0 + np.log(x[mask])
        x *= self.idf_                                   # (n, V) * (V,) 브로드캐스팅
        if self.norm == "l2":
            norms = np.linalg.norm(x, axis=1, keepdims=True)
            x /= np.where(norms > 0, norms, 1.0)          # 빈 문서는 0 벡터로 둔다
        return x

    def fit_transform(self, docs: list[list[str]]) -> np.ndarray:
        return self.fit(docs).transform(docs)
''',
    "src/search.py": r'''"""코사인 유사도 검색 (15~16장)."""
import numpy as np

from src.data import DOCS
from src.preprocess import preprocess
from src.tfidf import NumpyTfidf


class SearchEngine:
    def __init__(self, docs: list[str]):
        self.docs = docs
        self.vec = NumpyTfidf()
        self.X = self.vec.fit_transform([preprocess(d) for d in docs])   # 행마다 L2 정규화됨

    def search(self, query: str, topk: int = 5) -> list[tuple[float, int, str]]:
        if topk < 1:
            raise ValueError(f"topk는 1 이상이어야 합니다: {topk}")
        q = self.vec.transform([preprocess(query)])[0]
        if not q.any():
            return []
        scores = self.X @ q
        order = np.argsort(-scores, kind="stable")[:topk]
        return [(round(float(scores[i]), 3), int(i), self.docs[i][:40]) for i in order if scores[i] > 0]


_engine = None


def search(query: str, topk: int = 5) -> list[tuple[float, int, str]]:
    """과제의 (안 B) 시그니처: [(score, doc_id, text_snippet), ...]"""
    global _engine
    if _engine is None:
        _engine = SearchEngine(DOCS)
    return _engine.search(query, topk)
''',
    "main.py": r'''"""미니 검색기 CLI: python main.py --query "..." --topk 5"""
import argparse

from src.search import search


def main() -> None:
    ap = argparse.ArgumentParser(description="미니 TF-IDF 검색기")
    ap.add_argument("--query", required=True)
    ap.add_argument("--topk", type=int, default=5)
    args = ap.parse_args()
    if args.topk < 1:
        ap.error(f"--topk 는 1 이상이어야 합니다: {args.topk}")
    results = search(args.query, args.topk)
    print("=== 검색 결과 ===")
    print(f'Query: "{args.query}"\n')
    print(f"Top {len(results)} 결과:")
    for rank, (score, doc_id, snippet) in enumerate(results, start=1):
        print(f'{rank}. [{score:.3f}] doc {doc_id} "{snippet}..."')


if __name__ == "__main__":
    main()
''',
    "tests/test_tfidf.py": r'''import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

from src.tfidf import NumpyTfidf

DOCS = [["space", "rocket"], ["space"], ["game", "game", "team"]]


def test_idf_matches_hand_calculation():
    vec = NumpyTfidf().fit(DOCS)
    n = 3
    expected = {"space": np.log((1 + n) / (1 + 2)) + 1, "game": np.log((1 + n) / (1 + 1)) + 1}
    for word, value in expected.items():
        assert abs(vec.idf_[vec.vocabulary_[word]] - value) < 1e-12


def test_rows_are_unit_length():
    X = NumpyTfidf().fit_transform(DOCS)
    assert np.allclose(np.linalg.norm(X, axis=1), 1.0)


def test_unknown_words_are_ignored():
    vec = NumpyTfidf().fit(DOCS)
    assert not vec.transform([["moon"]]).any()


def test_matches_sklearn():
    ours = NumpyTfidf().fit_transform(DOCS)
    sk = TfidfVectorizer(analyzer=lambda d: d, lowercase=False).fit_transform(DOCS).toarray()
    assert np.abs(ours - sk).max() < 1e-6
''',
    "tests/test_search.py": r'''from src.search import search


def test_best_match_first():
    results = search("rocket space", topk=3)
    assert results[0][1] == 0                        # doc 0: rocket + space


def test_scores_sorted_descending():
    scores = [s for s, _, _ in search("baseball game")]
    assert scores == sorted(scores, reverse=True)


def test_stopword_only_query_returns_empty():
    assert search("the a in") == []
''',
}


MINI = Path(tempfile.mkdtemp()) / "mini-search"
for rel, content in MINI_SRC.items():
    path = MINI / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")

print("mini-search/")
for p in sorted(MINI.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(MINI)}  ({len(p.read_text(encoding='utf-8').splitlines())}줄)")

mini-search/
  main.py  (23줄)
  src/__init__.py  (0줄)
  src/data.py  (12줄)
  src/preprocess.py  (9줄)
  src/search.py  (34줄)
  src/tfidf.py  (58줄)
  tests/test_search.py  (15줄)
  tests/test_tfidf.py  (30줄)


파일 내용을 몇 개 확인합니다. `main.py`는 인자를 받아 `src.search.search()`를 부르고 결과를 출력하는 일**만** 합니다.

In [3]:
for rel in ["src/search.py", "main.py"]:
    print(f"── {rel} ──")
    print(MINI_SRC[rel])

── src/search.py ──
"""코사인 유사도 검색 (15~16장)."""
import numpy as np

from src.data import DOCS
from src.preprocess import preprocess
from src.tfidf import NumpyTfidf


class SearchEngine:
    def __init__(self, docs: list[str]):
        self.docs = docs
        self.vec = NumpyTfidf()
        self.X = self.vec.fit_transform([preprocess(d) for d in docs])   # 행마다 L2 정규화됨

    def search(self, query: str, topk: int = 5) -> list[tuple[float, int, str]]:
        if topk < 1:
            raise ValueError(f"topk는 1 이상이어야 합니다: {topk}")
        q = self.vec.transform([preprocess(query)])[0]
        if not q.any():
            return []
        scores = self.X @ q
        order = np.argsort(-scores, kind="stable")[:topk]
        return [(round(float(scores[i]), 3), int(i), self.docs[i][:40]) for i in order if scores[i] > 0]


_engine = None


def search(query: str, topk: int = 5) -> list[tuple[float, int, str]]:
    """과제의 (안 B) 시그니처: [(score, doc_id, text_snippet), ...]"""
    global _engine
   

### 실행 방법 (안 A: CLI)

In [4]:
run(["main.py", "--query", "space rocket", "--topk", "3"], MINI)
run(["main.py", "--query", "baseball game"], MINI)
run(["main.py", "--query", "space", "--topk", "0"], MINI)

$ python main.py --query space rocket --topk 3
=== 검색 결과 ===
Query: "space rocket"

Top 3 결과:
1. [0.827] doc 0 "The rocket launched into space!..."
2. [0.248] doc 1 "NASA launched a new space telescope...."
3. [0.219] doc 4 "Astronauts watched the baseball game in ..."

$ python main.py --query baseball game
=== 검색 결과 ===
Query: "baseball game"

Top 3 결과:
1. [0.647] doc 3 "Our team won the baseball game...."
2. [0.572] doc 4 "Astronauts watched the baseball game in ..."
3. [0.230] doc 5 "The hockey team lost the game in overtim..."



$ python main.py --query space --topk 0
usage: main.py [-h] --query QUERY [--topk TOPK]
main.py: error: --topk 는 1 이상이어야 합니다: 0



`--topk 0`처럼 잘못된 입력은 `argparse`의 `ap.error`로 사용법과 함께 거절합니다.

### 실행 방법 (안 B: 함수)

In [5]:
run(["-c", "from src.search import search; print(search('telescope image', topk=3))"], MINI)

$ python -c from src.search import search; print(search('telescope image', topk=3))
[(0.678, 6, 'A new telescope image of the galaxy was '), (0.315, 1, 'NASA launched a new space telescope.')]



반환 형식이 과제의 `search(query: str) -> List[(score, doc_id, text_snippet)]`와 같습니다.

### 실제 프로젝트의 `main.py`
실제 프로젝트는 같은 구조에 `--mode`로 기능을 고릅니다. **프로젝트 폴더에서** 실행합니다.

| 명령 | 하는 일 | 결과물 |
|---|---|---|
| `uv run python main.py --query "space shuttle launch" --topk 5` | 영어 검색 (안 A) | 화면 출력 |
| `uv run python main.py --lang ko --query "삼성전자 반도체 실적" --explain` | 한국어 검색 + 1위 문서 계산 과정 | 화면 출력 |
| `uv run python main.py --mode verify --lang en` | TF-IDF vs sklearn 검증 | `outputs/en/verify_log.txt` |
| `uv run python main.py --mode classify --lang ko` | 분류, 혼동 행렬, 오분류 | `outputs/ko-okt/` 아래 3개 파일 |
| `uv run python main.py --mode bow` | BoW 한계 사례 (21장) | 화면 출력 |
| `uv run python main.py --mode compare` | 한국어 Okt vs 공백 (9장) | 화면 출력 |
| `uv run python main.py --mode errors` | 길이별 정확도, 레이블 모호성 (20장) | 화면 출력 |
| `uv run python main.py --mode all` | 위 전부 | `outputs/` 전체 다시 생성 |

실제 프로젝트의 모듈 구성을 읽어서 확인해 봅니다(파일을 바꾸지 않습니다).

In [6]:
for path in sorted((PROJECT / "src").glob("*.py")):
    if path.name == "__init__.py":
        continue
    tree = ast.parse(path.read_text(encoding="utf-8"))
    doc = (ast.get_docstring(tree) or "").splitlines()[0] if ast.get_docstring(tree) else ""
    funcs = [n.name for n in tree.body if isinstance(n, (ast.FunctionDef, ast.ClassDef)) and not n.name.startswith("_")]
    print(f"src/{path.name:<22} {doc[:45]}")
    print(f"{'':27}공개 함수·클래스: {', '.join(funcs[:6])}{' …' if len(funcs) > 6 else ''}")

src/bow_limits.py          BoW(TF-IDF) 한계를 실제 벡터로 보여 준다. 말뭉치로 학습한 어휘·IDF
                           공개 함수·클래스: pair_report, main
src/classify.py            TF-IDF 벡터 기반 문서 분류 (A4-1): Logistic Regressio
                           공개 함수·클래스: ClassificationResult, split, run_classification, display_label, plot_confusion_matrix, top_contributions …
src/compare_tokenizers.py  한국어 토큰화 비교: Okt 형태소 분석 vs 공백 토큰화 (같은 7,000건, 
                           공개 함수·클래스: corpus_stats, main
src/corpus.py              데이터 로드 + 전처리를 한 번에: 문서마다 토큰 리스트를 붙이고, 토큰이 너무 
                           공개 함수·클래스: output_dir, load_corpus
src/data.py                데이터 로드 (A4-1): 영어 20 Newsgroups 5개 카테고리, 한국어 
                           공개 함수·클래스: mask_pii, drop_duplicates, load_english, download_ynat, load_korean, load_dataset
src/error_analysis.py      오분류 원인을 숫자로 확인한다.
                           공개 함수·클래스: accuracy_by_length, korean_label_ambiguity, main
src/preprocess.py          전처리 파이프라인 (A4-1): 정제 → 토큰화 → 불용어 제거.

## 22.2 결과물 저장 규칙

평가자가 결과를 찾기 쉽도록 **데이터셋별 폴더에 같은 이름**으로 저장합니다. 코드를 다시 실행하면 같은 파일이 덮어써지므로, 결과물은 항상 현재 코드와 일치합니다.

```
outputs/
├── en/              # 영어 20 Newsgroups
│   ├── verify_log.txt            # 14장: TF-IDF 검증 로그
│   ├── classification_report.txt # 19장: 정확도, F1, 클래스별 지표
│   ├── confusion_matrix.png      # 19장: 혼동 행렬
│   └── misclassified.txt         # 20장: 오분류 사례
├── ko-okt/          # 한국어 KLUE-YNAT, Okt 토큰화
└── ko-whitespace/   # 한국어, 공백 토큰화 (비교용)
```

- 폴더 이름에 **데이터와 토큰화 방식**을 함께 넣어, 같은 실험을 설정만 바꿔 돌려도 결과가 섞이지 않게 합니다.
- 파일 맨 위에 **데이터, 설정, 환경(버전)**을 적어 두면 파일 하나만 보고도 재현할 수 있습니다.

실제 프로젝트의 결과물을 읽어 봅니다.

In [7]:
for sub in ["en", "ko-okt", "ko-whitespace"]:
    files = sorted(p.name for p in (PROJECT / "outputs" / sub).glob("*"))
    print(f"outputs/{sub}/: {files}")

print("\n── outputs/en/verify_log.txt (앞 12줄) ──")
print("\n".join((PROJECT / "outputs/en/verify_log.txt").read_text(encoding="utf-8").splitlines()[:12]))
print("\n── outputs/ko-okt/classification_report.txt (앞 8줄) ──")
print("\n".join((PROJECT / "outputs/ko-okt/classification_report.txt").read_text(encoding="utf-8").splitlines()[:8]))

outputs/en/: ['classification_report.txt', 'confusion_matrix.png', 'misclassified.txt', 'verify_log.txt']
outputs/ko-okt/: ['classification_report.txt', 'confusion_matrix.png', 'misclassified.txt', 'verify_log.txt']
outputs/ko-whitespace/: ['classification_report.txt', 'confusion_matrix.png', 'misclassified.txt', 'verify_log.txt']

── outputs/en/verify_log.txt (앞 12줄) ──
=== TF-IDF 구현 검증 결과 ===
데이터: en, 문서 4,646건
환경: Python 3.13.5, NumPy 2.5.3, scikit-learn 1.9.1
공통 sklearn 설정: TfidfVectorizer(analyzer=<전처리된 토큰 그대로>, lowercase=False, min_df=3, dtype=float64)

[검증 1: 기본 설정] 직접 구현 vs Scikit-learn
  - 직접 구현 설정: NumpyTfidf(tf='raw', smooth_idf=True, norm='l2', min_df=3)
  - sklearn 설정 : sublinear_tf=False, smooth_idf=True, norm='l2'
  - TF-IDF 행렬 shape: (4646, 13387), 어휘 일치: True, 0이 아닌 위치 일치: True
  - IDF 최대 오차: 0.00e+00
  - 최대 오차: 1.17e-15
  - 평균 오차: 4.96e-20

── outputs/ko-okt/classification_report.txt (앞 8줄) ──
=== 분류 성능 평가 ===
데이터: ko-okt
모델: LogisticRegression(C=1.0, max_iter=2000, r

## 22.3 테스트로 결과 지키기

전처리 규칙이나 TF-IDF 코드를 고치다 보면 **다른 곳이 조용히 깨질** 수 있습니다. 테스트는 "이건 항상 이래야 한다"를 코드로 적어 두고, 고칠 때마다 자동으로 확인합니다.

이 과제에서 테스트할 만한 것:

| 대상 | 확인할 것 |
|---|---|
| TF-IDF | 손계산 IDF와 일치, 행 길이 1, OOV 무시, sklearn과 1e-6 이내 |
| 검색 | 점수 내림차순, topk 개수, topk < 1 거부, 불용어만 있는 쿼리 → 빈 결과 |
| 전처리 | 소문자·소유격, 불용어, 숫자, 한자 유지 (10장 테스트) |
| 분류 | Train에만 fit하는지, 분할이 재현되는지 |

미니 프로젝트의 테스트를 실행합니다. `python -m pytest`로 실행하면 현재 폴더가 import 경로에 들어가서 `from src...`가 동작합니다(1장).

In [8]:
run(["-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"], MINI)

$ python -m pytest -q --color=no -p no:cacheprovider
.......                                                                  [100%]
7 passed in 1.10s



### 일부러 망가뜨려 보기
IDF 식 끝의 `+ 1.0`을 지운 "실수"를 넣고 테스트를 다시 돌려 봅니다.

In [9]:
tfidf_path = MINI / "src/tfidf.py"
original = tfidf_path.read_text(encoding="utf-8")
broken = original.replace("np.log((1.0 + n) / (1.0 + self.df_)) + 1.0", "np.log((1.0 + n) / (1.0 + self.df_))")
assert broken != original
tfidf_path.write_text(broken, encoding="utf-8")

result = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"],
                        cwd=MINI, capture_output=True, text=True, env=ENV)
print("\n".join(l for l in result.stdout.splitlines() if l.startswith(("FAILED", "PASSED")) or "passed" in l or "failed" in l))

tfidf_path.write_text(original, encoding="utf-8")   # 되돌리기
run(["-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"], MINI)

FAILED tests/test_tfidf.py::test_idf_matches_hand_calculation - assert np.flo...
FAILED tests/test_tfidf.py::test_matches_sklearn - AssertionError: assert np....
2 failed, 5 passed in 0.96s


$ python -m pytest -q --color=no -p no:cacheprovider
.......                                                                  [100%]
7 passed in 1.04s



손계산 IDF 테스트와 sklearn 비교 테스트가 바로 실패를 알려 줍니다. 되돌리면 다시 통과합니다. 실제 프로젝트에서는 `uv run pytest tests -q`로 실행합니다. 프로젝트 테스트는 데이터 파일 없이 작은 가짜 말뭉치로 돌도록 만들어서 빠르고, 어느 컴퓨터에서나 돌아갑니다.

In [10]:
for path in sorted((PROJECT / "tests").glob("test_*.py")):
    names = re.findall(r"^def (test_\w+)", path.read_text(encoding="utf-8"), flags=re.M)
    print(f"tests/{path.name:<20} 테스트 함수 {len(names)}개: {', '.join(names[:3])}{' …' if len(names) > 3 else ''}")

tests/test_classify.py     테스트 함수 3개: test_split_8_2_stratified, test_vectorizer_fit_on_train_only, test_metrics_and_misclassified_table
tests/test_preprocess.py   테스트 함수 9개: test_mask_pii_hides_email_url_phone, test_english_pipeline, test_english_drops_long_garbage_tokens …
tests/test_search.py       테스트 함수 8개: test_cosine_matches_definition, test_cosine_zero_query, test_top_k_sorted_desc …
tests/test_tfidf.py        테스트 함수 9개: test_vocabulary_is_sorted_and_df, test_counts_and_smooth_idf_by_hand, test_l2_rows_have_unit_norm …


## 22.4 README 필수 3단 템플릿

과제는 README를 **구현 요약 → 검증/실험 결과 → 한계와 개선 방향** 세 부분으로 쓰라고 요구하고, **실행 방법**(안 A 또는 B)을 반드시 넣으라고 합니다. 아래 템플릿을 복사해서 각 장의 📝 리포트용 정리를 채워 넣으면 됩니다. `<...>`는 실제 값으로 바꿉니다.

````markdown
# A4-1. 문서 검색·분류 시스템 (TF-IDF 직접 구현)

## 한눈에 보기
| 질문 | 답 |
|---|---|
| 무엇을 만들었나 | <데이터>로 TF-IDF 검색·분류 시스템. TF-IDF와 코사인 유사도는 NumPy로 직접 구현 |
| 검증 결과 | sklearn TfidfVectorizer와 최대 오차 <값> (허용 1e-6) |
| 분류 성능 | 정확도 <값> / macro F1 <값> |
| 핵심 한계 | <어순·부정·동의어 사례 한 줄> |

## 실행 방법
```bash
pip install -r requirements.txt        # 또는 uv sync
# (안 A) CLI
python main.py --query "space shuttle launch" --topk 5
# 검증·분류
python main.py --mode verify --lang en
python main.py --mode classify --lang en
# 테스트
pytest tests -q
```
(안 B) 함수: `search(query: str, topk: int = 5) -> list[tuple[float, str, str]]`
예시 출력: <실제 출력 붙여넣기>

## 1. 구현 요약
### 1.1 데이터          (7장) 출처, 라이선스, 문서 수, 카테고리, 개인정보 처리, 정리 단계별 건수
### 1.2 전처리 파이프라인 (8~10장) 정제·토큰화·불용어 기준표와 설계 근거
### 1.3 TF-IDF 수식과 구현 (12~13장) 수식, 각 +1의 의미, 구현 단계
### 1.4 검색 시스템      (15~16장) 쿼리 처리, 코사인 유사도, top-k, 스니펫
### 1.5 분류 시스템      (17~18장) 모델, 분할, Train에서만 fit

## 2. 검증·실험 결과
### 2.1 TF-IDF 검증 로그 (14장) sklearn 설정 명시 + 로그
### 2.2 검색 결과        (16장) 예시 쿼리와 결과, 실패 쿼리
### 2.3 분류 성능        (19장) 정확도, F1, 클래스별 지표, 혼동 행렬 그림
### 2.4 토큰화 비교      (9장)  한국어 공백 vs 형태소: 성능이 왜 달라지는가

## 3. 한계와 개선 방향
### 3.1 오분류 사례 분석 (20장) 5건 이상: 예측/실제/문서/원인/한계
### 3.2 BoW 한계 분석    (21장) 어순, 동음이의어, 동의어, 부정, OOV 사례표
### 3.3 개선 방향과 후속 학습 (21장)

## 4. 테스트
````

### 쓰는 요령
- **숫자는 결과 파일에서 복사**합니다. 코드를 고쳐 결과가 바뀌면 README 숫자도 다시 맞춥니다.
- 설계 결정마다 **"무엇을, 왜, 그 대가는"**을 함께 씁니다(8~10장 리포트용 정리 표 형식).
- 검증 로그에는 **sklearn 설정을 명시**합니다. 과제가 따로 요구하는 항목입니다(14장).
- 좋은 결과만 쓰지 않습니다. 누수 실험의 작은 차이(17장), 실패 쿼리(16장), 형태소 오분석(9장)처럼 **기대와 다른 결과**도 그대로 쓰면 오히려 신뢰를 얻습니다.

## 22.5 재현성

다른 사람이 같은 명령으로 **같은 숫자**를 얻을 수 있어야 합니다.

| 항목 | 방법 |
|---|---|
| 난수 | 분할, 샘플링, 모델의 `random_state`를 한 상수(`RANDOM_STATE = 42`)로 고정 |
| 데이터 | 출처 URL, 받는 방법, 샘플링 규칙을 README와 코드에 명시 |
| 설정 | `min_df`, 불용어 목록, sklearn 검증 설정을 상수와 로그에 기록 |
| 버전 | `requirements.txt`에 `==`로 고정, 로그 맨 위에 Python·NumPy·sklearn 버전 출력 |
| 캐시 | 형태소 분석 캐시 이름에 설정 지문(9장)을 넣어 오래된 결과를 쓰지 않게 |

`requirements.txt`의 버전이 지금 환경과 같은지 확인합니다.

In [11]:
print("Python", sys.version.split()[0])
for line in (PROJECT / "requirements.txt").read_text(encoding="utf-8").splitlines():
    m = re.match(r"([A-Za-z0-9_.-]+)==(\S+)", line.strip())
    if m:
        name, pinned = m.groups()
        installed = version(name)
        print(f"{'✅' if installed == pinned else '⚠️'} {name:<13} requirements {pinned:<8} 설치됨 {installed}")

Python 3.13.5
✅ numpy         requirements 2.5.3    설치됨 2.5.3
✅ pandas        requirements 3.0.6    설치됨 3.0.6
✅ scikit-learn  requirements 1.9.1    설치됨 1.9.1
✅ matplotlib    requirements 3.11.2   설치됨 3.11.2
✅ nltk          requirements 3.10.3   설치됨 3.10.3
✅ konlpy        requirements 0.6.0    설치됨 0.6.0
✅ pytest        requirements 9.1.1    설치됨 9.1.1


## 22.6 제출 전 체크리스트

과제 문서의 요구사항을 모두 항목으로 만들고, 실제 프로젝트 폴더를 **읽기만 해서** 확인합니다. 자동으로 확인할 수 없는 항목(근거 서술의 질 등)은 직접 읽고 체크합니다.

In [12]:
readme = (PROJECT / "README.md").read_text(encoding="utf-8")
outputs = PROJECT / "outputs"
SUBS = ["en", "ko-okt", "ko-whitespace"]


def imports_sklearn(rel: str) -> bool:
    tree = ast.parse((PROJECT / rel).read_text(encoding="utf-8"))
    names = [a.name for n in ast.walk(tree) if isinstance(n, ast.Import) for a in n.names]
    names += [n.module or "" for n in ast.walk(tree) if isinstance(n, ast.ImportFrom)]
    return any(name.startswith("sklearn") for name in names)


verify_logs = [(outputs / s / "verify_log.txt").read_text(encoding="utf-8") for s in SUBS]
doc_counts = [int(m.replace(",", "")) for log in verify_logs for m in re.findall(r"문서 ([\d,]+)건", log)[:1]]
n_cases = len(re.findall(r"^\[", (outputs / "en/misclassified.txt").read_text(encoding="utf-8"), flags=re.M))
n_tests = sum(len(re.findall(r"^def test_", p.read_text(encoding="utf-8"), flags=re.M))
              for p in (PROJECT / "tests").glob("test_*.py"))
gitignore = (PROJECT.parents[1] / ".gitignore").read_text(encoding="utf-8")

checks = [
    ("데이터: 문서 500개 이상", all(c >= 500 for c in doc_counts), f"{doc_counts}"),
    ("TF-IDF 핵심 로직에 sklearn 미사용 (src/tfidf.py)", not imports_sklearn("src/tfidf.py"), ""),
    ("코사인 유사도에 sklearn 미사용 (src/search.py)", not imports_sklearn("src/search.py"), ""),
    ("검증 로그: 모두 PASS, FAIL 없음", all("PASS" in l and "FAIL" not in l for l in verify_logs),
     f"PASS {sum(l.count('PASS') for l in verify_logs)}개"),
    ("검증 로그: sklearn 설정 명시", all("smooth_idf" in l and "sublinear_tf" in l and "norm" in l for l in verify_logs), ""),
    ("분류 리포트: 정확도와 F1", all("정확도" in (outputs / s / "classification_report.txt").read_text(encoding="utf-8")
                               and "F1" in (outputs / s / "classification_report.txt").read_text(encoding="utf-8") for s in SUBS), ""),
    ("혼동 행렬 그림", all((outputs / s / "confusion_matrix.png").exists() for s in SUBS), ""),
    ("오분류 사례 5건 이상 (영어 파일)", n_cases >= 5, f"{n_cases}건"),
    ("README: 구현 요약 / 검증·실험 / 한계와 개선", all(k in readme for k in ["구현 요약", "검증", "한계와 개선"]), ""),
    ("README: 실행 방법 (안 A 또는 B)", ("--query" in readme and "--topk" in readme) or "search(" in readme, ""),
    ("README: sklearn 검증 설정 명시", all(k in readme for k in ["smooth_idf", "sublinear_tf", "norm"]), ""),
    ("README: 혼동 행렬, BoW 한계, 오분류 원인", all(k in readme for k in ["혼동 행렬", "BoW", "오분류"]), ""),
    ("requirements.txt 존재", (PROJECT / "requirements.txt").exists(), ""),
    ("main.py에 --query, --topk", all(k in (PROJECT / "main.py").read_text(encoding="utf-8") for k in ["--query", "--topk"]), ""),
    ("테스트 존재", n_tests > 0, f"테스트 함수 {n_tests}개"),
    ("원본 데이터·캐시는 git 제외", "a4-1-document-search/data/raw" in gitignore and "a4-1-document-search/data/cache" in gitignore, ""),
]
for name, ok, detail in checks:
    print(f"{'✅' if ok else '❌'} {name}" + (f"  ({detail})" if detail else ""))
print(f"\n자동 점검 {sum(ok for _, ok, _ in checks)}/{len(checks)} 통과")

✅ 데이터: 문서 500개 이상  ([4646, 7000, 6998])
✅ TF-IDF 핵심 로직에 sklearn 미사용 (src/tfidf.py)
✅ 코사인 유사도에 sklearn 미사용 (src/search.py)
✅ 검증 로그: 모두 PASS, FAIL 없음  (PASS 12개)
✅ 검증 로그: sklearn 설정 명시
✅ 분류 리포트: 정확도와 F1
✅ 혼동 행렬 그림
✅ 오분류 사례 5건 이상 (영어 파일)  (93건)
✅ README: 구현 요약 / 검증·실험 / 한계와 개선
✅ README: 실행 방법 (안 A 또는 B)
✅ README: sklearn 검증 설정 명시
✅ README: 혼동 행렬, BoW 한계, 오분류 원인
✅ requirements.txt 존재
✅ main.py에 --query, --topk
✅ 테스트 존재  (테스트 함수 29개)
✅ 원본 데이터·캐시는 git 제외

자동 점검 16/16 통과


### 직접 읽고 확인할 항목

- [ ] 전처리 기준(특수문자 범위, 불용어, 동의어)의 **근거**가 README에 있다 (학습 목표 ⑤)
- [ ] 한국어 공백 토큰화와 형태소 분석의 차이를 **"왜"**까지 설명했다 (9장)
- [ ] TF, IDF, TF-IDF 수식과 각 항의 역할이 있다 (학습 목표 ②)
- [ ] 코사인 유사도 계산 과정 예시가 있다 (학습 목표 ③)
- [ ] 오분류 사례마다 원인과 **BoW 한계**를 연결했다 (학습 목표 ④)
- [ ] 개인정보가 결과물(스니펫, 오분류 사례)에 노출되지 않는다
- [ ] 데이터 출처와 라이선스를 적었다
- [ ] README의 숫자가 `outputs/` 파일과 일치한다
- [ ] 새로 받은 환경에서 README의 명령만으로 실행된다

### 평가 대비
평가에서는 결과물보다 **설명**을 많이 묻습니다. 부록 A(평가 대비 질문 모음)의 질문에 자료를 보지 않고 답해 보세요. 막히는 질문은 해당 장의 퀴즈와 📝 리포트용 정리로 돌아가면 됩니다.

## ✍️ 실습: 테스트 추가하기

미니 프로젝트에 `tests/test_topk.py`를 추가하세요.
1. `search("space", topk=0)`이 `ValueError`를 내는지 확인하는 테스트 (`pytest.raises` 사용)
2. `search("game", topk=2)`가 최대 2개를 돌려주는지 확인하는 테스트

파일을 쓴 뒤 `run(["-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"], MINI)`로 실행해서 전체가 통과하는지 확인합니다.

In [13]:
# 여기에 작성하세요

**정답**

In [14]:
(MINI / "tests/test_topk.py").write_text(
    "import pytest\n"
    "\n"
    "from src.search import search\n"
    "\n"
    "\n"
    "def test_topk_below_one_is_rejected():\n"
    "    with pytest.raises(ValueError):\n"
    "        search('space', topk=0)\n"
    "\n"
    "\n"
    "def test_topk_limits_result_count():\n"
    "    assert len(search('game', topk=2)) <= 2\n",
    encoding="utf-8",
)
run(["-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"], MINI)

$ python -m pytest -q --color=no -p no:cacheprovider
.........                                                                [100%]
9 passed in 1.01s



## 🧐 퀴즈

**Q1.** 노트북에서 모든 걸 구현해도 되는데, `src/` 모듈과 `main.py`로 나누는 이유는?
<details><summary>▶ 정답 보기</summary>

평가자가 명령 한 줄로 실행할 수 있어야 하고, 검색·분류·검증·테스트가 **같은 코드**를 불러 써야 결과가 일치하기 때문입니다. 노트북은 셀 실행 순서에 따라 결과가 달라질 수 있고 다른 코드에서 import하기 어렵습니다.
</details>

**Q2.** `python -m pytest`로 실행하면 `from src.search import search`가 동작하는 이유는?
<details><summary>▶ 정답 보기</summary>

`python -m`은 현재 폴더(프로젝트 루트)를 import 경로에 넣기 때문에, 루트 아래의 `src` 패키지를 찾을 수 있습니다(1장 1.3절).
</details>

**Q3.** 결과 파일 맨 위에 데이터, 설정, 라이브러리 버전을 적는 이유는?
<details><summary>▶ 정답 보기</summary>

파일 하나만 보고도 어떤 조건에서 나온 결과인지 알 수 있고, 다른 환경에서 숫자가 다르게 나왔을 때 원인(버전, 설정 차이)을 찾을 수 있기 때문입니다.
</details>

**Q4.** README 3단 구성은 무엇이고, 각 부분에 이 자료의 어느 장 내용이 들어가나요?
<details><summary>▶ 정답 보기</summary>

① 구현 요약: 데이터(7장), 전처리(8~10장), TF-IDF(12~13장), 검색(15~16장), 분류(17~18장). ② 검증·실험 결과: 검증 로그(14장), 검색 결과(16장), 분류 성능과 혼동 행렬(19장), 토큰화 비교(9장). ③ 한계와 개선 방향: 오분류 분석(20장), BoW 한계와 개선 방향(21장).
</details>

**Q5.** IDF 식을 고친 뒤 테스트가 모두 통과했다면, README에서 무엇을 다시 확인해야 하나요?
<details><summary>▶ 정답 보기</summary>

결과 숫자입니다. 테스트 통과는 "코드가 정의대로 동작한다"는 뜻일 뿐, 검증 오차·분류 성능·사례 수치는 달라졌을 수 있습니다. `main.py --mode all`로 결과를 다시 만들고 README의 숫자를 결과 파일과 대조합니다.
</details>

## ✅ 체크포인트
- [ ] 노트북 코드를 단계별 모듈과 `main.py`로 나눌 수 있다.
- [ ] 결과물을 데이터셋별 폴더에 설정과 함께 저장할 수 있다.
- [ ] 테스트를 쓰고, 일부러 망가뜨려서 테스트가 잡아내는지 확인할 수 있다.
- [ ] README 3단 구성과 실행 방법을 채울 수 있다.
- [ ] 제출 전 체크리스트를 모두 통과했다.

## 🔗 과제 연결
- 이 장으로 과제의 최종 결과물 4가지(전처리·TF-IDF 코드와 검증 로그, 검색·분류 시스템, README 3단 리포트, 실행 방법)를 모두 갖췄습니다.
- 이제 **부록 A(평가 대비 질문 모음)**로 학습 목표 5개를 말로 설명하는 연습을 하세요. 막히는 에러는 **부록 C**, 헷갈리는 용어는 **부록 B**를 참고합니다.